# 04 — Contamination Sensitivity

Sweep the contamination parameter from 0.001 to 0.10 for each detector on each dataset. Show that F1 is highly sensitive to this choice, and that Mahalanobis (no contamination parameter) is more robust.

In [1]:
import sys
from pathlib import Path

# Ensure project root is on sys.path regardless of cwd
_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import json
import warnings

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURES = _ROOT / "outputs" / "figures"
OUTPUTS = _ROOT / "outputs"
FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

from src.datasets import load_network, load_manufacturing, make_synthetic_credit_card
from src.detectors import (
    IsolationForestDetector,
    LOFDetector,
    OneClassSVMDetector,
    DBSCANDetector,
    MahalanobisDetector,
    AutoencoderDetector,
)
from src.evaluation import (
    run_arena,
    sweep_contamination,
    evaluate_detector,
    auprc,
    precision_recall_f1_at_contamination,
)
import src.visualisation as vis


## Load datasets

In [2]:
X_cc, y_cc = make_synthetic_credit_card(n_samples=5000, random_state=42)
X_net, y_net = load_network(n_samples=3000, random_state=42)
X_mfg, y_mfg = load_manufacturing(n_samples=2000, random_state=42)

TRUE_CONTAMINATION = {"credit_card": 0.0017, "network": 0.02, "manufacturing": 0.015}
CONTAMINATION_RANGE = np.linspace(0.001, 0.10, 30)
print(f"Sweep range: {CONTAMINATION_RANGE[0]:.3f} → {CONTAMINATION_RANGE[-1]:.3f}  ({len(CONTAMINATION_RANGE)} steps)")


Sweep range: 0.001 → 0.100  (30 steps)


## Run contamination sweep — network dataset

In [3]:
detectors_sweep = [
    IsolationForestDetector(n_estimators=100, random_state=42),
    LOFDetector(n_neighbors=20),
    MahalanobisDetector(),
    AutoencoderDetector(hidden_dim=32, epochs=20, random_state=42),
]

split_net = int(0.8 * len(X_net))
X_tr_net, X_te_net, y_te_net = X_net[:split_net], X_net[split_net:], y_net[split_net:]

sweep_results = {}
for det in detectors_sweep:
    df = sweep_contamination(det, X_tr_net, X_te_net, y_te_net, CONTAMINATION_RANGE)
    sweep_results[det.name] = df
    print(f"  {det.name}: peak F1={df['f1'].max():.3f} at contamination={df.loc[df['f1'].idxmax(), 'contamination']:.4f}")


  IsolationForest: peak F1=0.952 at contamination=0.0181


  LOF: peak F1=0.400 at contamination=0.0078
  Mahalanobis: peak F1=0.952 at contamination=0.0181


  Autoencoder: peak F1=0.952 at contamination=0.0181


## F1 sensitivity plot — network dataset

In [4]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, df in sweep_results.items():
    ax.plot(df["contamination"], df["f1"], marker="o", ms=3, lw=1.5, label=name)
ax.axvline(TRUE_CONTAMINATION["network"], color="k", ls="--", lw=1, label=f"true contamination ({TRUE_CONTAMINATION['network']})")
ax.set_xlabel("Contamination rate")
ax.set_ylabel("F1 score")
ax.set_title("Network Dataset — F1 vs Contamination Rate")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(FIGURES / "04_contamination_f1_network.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved 04_contamination_f1_network.png")


Saved 04_contamination_f1_network.png


## Precision and recall separately

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, df in sweep_results.items():
    axes[0].plot(df["contamination"], df["precision"], lw=1.2, label=name)
    axes[1].plot(df["contamination"], df["recall"], lw=1.2, label=name)
for ax, metric in zip(axes, ["Precision", "Recall"]):
    ax.axvline(TRUE_CONTAMINATION["network"], color="k", ls="--", lw=1)
    ax.set_xlabel("Contamination rate")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs Contamination — Network")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(FIGURES / "04_contamination_pr_network.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved 04_contamination_pr_network.png")


Saved 04_contamination_pr_network.png


## Sweep all three datasets for Isolation Forest

In [6]:
datasets_sweep = {
    "credit_card": (X_cc, y_cc, TRUE_CONTAMINATION["credit_card"]),
    "network": (X_net, y_net, TRUE_CONTAMINATION["network"]),
    "manufacturing": (X_mfg, y_mfg, TRUE_CONTAMINATION["manufacturing"]),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
all_sweep_data = {}

for ax, (ds_name, (X, y, true_cont)) in zip(axes, datasets_sweep.items()):
    split = int(0.8 * len(X))
    X_tr, X_te, y_te = X[:split], X[split:], y[split:]
    det_if = IsolationForestDetector(n_estimators=100, random_state=42)
    df = sweep_contamination(det_if, X_tr, X_te, y_te, CONTAMINATION_RANGE)
    all_sweep_data[ds_name] = {"IsolationForest": df.to_dict(orient="records")}

    ax.plot(df["contamination"], df["f1"], color="steelblue", lw=1.5, label="F1")
    ax.plot(df["contamination"], df["precision"], color="orange", lw=1.2, ls="--", label="Precision")
    ax.plot(df["contamination"], df["recall"], color="tomato", lw=1.2, ls=":", label="Recall")
    ax.axvline(true_cont, color="k", ls="--", lw=1, label=f"true={true_cont}")
    ax.set_xlabel("Contamination")
    ax.set_title(f"Isolation Forest — {ds_name}")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7)

fig.suptitle("Isolation Forest: F1 sensitivity to contamination rate", fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES / "04_if_contamination_all_datasets.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved 04_if_contamination_all_datasets.png")


Saved 04_if_contamination_all_datasets.png


## F1 range (sensitivity metric)

In [7]:
print("Isolation Forest F1 range across contamination sweep:")
for ds_name in datasets_sweep:
    records = all_sweep_data[ds_name]["IsolationForest"]
    f1_vals = [r["f1"] for r in records]
    f1_range = max(f1_vals) - min(f1_vals)
    print(f"  {ds_name:15s}: min={min(f1_vals):.3f}  max={max(f1_vals):.3f}  range={f1_range:.3f}")
print()
print("High range → contamination choice critically affects performance.")


Isolation Forest F1 range across contamination sweep:
  credit_card    : min=0.020  max=1.000  range=0.980
  network        : min=0.182  max=0.952  range=0.771
  manufacturing  : min=0.222  max=1.000  range=0.778

High range → contamination choice critically affects performance.


## Save sweep results → `outputs/04_contamination_sweep.json`

In [8]:
sweep_json = {}
for ds_name, (X, y, true_cont) in datasets_sweep.items():
    sweep_json[ds_name] = {"true_contamination": true_cont, "detectors": {}}
    split = int(0.8 * len(X))
    X_tr, X_te, y_te = X[:split], X[split:], y[split:]
    for det in [IsolationForestDetector(n_estimators=100, random_state=42), MahalanobisDetector()]:
        df = sweep_contamination(det, X_tr, X_te, y_te, CONTAMINATION_RANGE)
        f1_vals = df["f1"].tolist()
        sweep_json[ds_name]["detectors"][det.name] = {
            "contamination_range": df["contamination"].tolist(),
            "f1": [round(v, 4) for v in f1_vals],
            "precision": [round(v, 4) for v in df["precision"].tolist()],
            "recall": [round(v, 4) for v in df["recall"].tolist()],
            "peak_f1": round(float(max(f1_vals)), 4),
            "f1_range": round(float(max(f1_vals) - min(f1_vals)), 4),
        }

with (OUTPUTS / "04_contamination_sweep.json").open("w") as f:
    json.dump(sweep_json, f, indent=2)
print("Saved outputs/04_contamination_sweep.json")


Saved outputs/04_contamination_sweep.json
